# CRM Access Governance & Customer Data Protection
## Stage 7 — Data Lineage & Traceability

This notebook documents how data moves from source systems through transformations, governance controls, privacy controls, and analytical outputs.

### Main Goals
1. Define data layers and source objects.
2. Document source-to-target mappings.
3. Build field-level lineage.
4. Register transformations.
5. Link fields to Data Quality rules.
6. Link fields to governance rules.
7. Link customer fields to privacy controls.
8. Trace governed fields into analytical outputs.
9. Build end-to-end traceability matrices.
10. Support future audit and impact analysis.


In [1]:
# 1. Libraries and Settings
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)

CRM_SOURCE = "Permission_Aware_CRM_Governance_Synthetic_50000.csv"
CUSTOMER_SOURCE = "Synthetic Customer Layer (Stage 6)"

print("Environment ready.")


Environment ready.


# 2. Core Lineage Concepts

- **Source-to-Target Mapping:** how a source field becomes a target field.
- **Transformation:** the logic applied between source and target.
- **Field-Level Lineage:** relationships between individual fields across layers.
- **Business Lineage:** relationship between data, controls, KPIs, and business use.
- **Traceability:** the ability to explain a path from source to rule, transformation, and output.


# 3. Proposed Data Architecture

```text
CRM ACCESS SOURCE
      |
      v
RAW ACCESS EVENTS
      |
      +--> DATA QUALITY RULES
      +--> ACCESS GOVERNANCE RULES
      |
      v
GOVERNED ACCESS EVENTS
      |
      v
ANALYTICAL ACCESS MODEL
      |
      v
POWER BI GOVERNANCE MONITORING


CUSTOMER CRM SOURCE
      |
      v
RAW CUSTOMER DATA
      |
      +--> PRIVACY CLASSIFICATION
      +--> MINIMIZATION / MASKING / PSEUDONYMIZATION
      |
      v
ANALYTICS-SAFE CUSTOMER LAYER
```


# 4. Layer Inventory

In [2]:
layer_inventory = pd.DataFrame([
    ["SRC-001","Source","CRM Access Source",CRM_SOURCE,"Synthetic access-event source"],
    ["RAW-001","Raw","Raw Access Events","raw_access_events","Unmodified CRM access-event layer"],
    ["GOV-001","Governed","Governed Access Events","governed_access_events","Events enriched with quality and governance logic"],
    ["ANA-001","Analytical","Access Analytical Model","fact_access_event","Power BI-ready access governance layer"],
    ["SRC-002","Source","Customer CRM Source",CUSTOMER_SOURCE,"Synthetic customer source created in Stage 6"],
    ["RAW-002","Raw","Raw Customer Data","raw_customer","Customer layer containing direct identifiers"],
    ["PRV-001","Privacy","Protected Customer Layer","protected_customer","Customer data after privacy transformations"],
    ["ANA-002","Analytical","Analytics-Safe Customer Layer","analytics_safe_customer","Customer layer without unnecessary direct identifiers"]
], columns=["Layer_ID","Layer_Type","Layer_Name","Object_Name","Description"])
display(layer_inventory)


,Layer_ID,Layer_Type,Layer_Name,Object_Name,Description
0,SRC-001,Source,CRM Access Source,Permission_Aware_CRM_Governance_Synthetic_5000...,Synthetic access-event source
1,RAW-001,Raw,Raw Access Events,raw_access_events,Unmodified CRM access-event layer
2,GOV-001,Governed,Governed Access Events,governed_access_events,Events enriched with quality and governance logic
3,ANA-001,Analytical,Access Analytical Model,fact_access_event,Power BI-ready access governance layer
4,SRC-002,Source,Customer CRM Source,Synthetic Customer Layer (Stage 6),Synthetic customer source created in Stage 6
5,RAW-002,Raw,Raw Customer Data,raw_customer,Customer layer containing direct identifiers
6,PRV-001,Privacy,Protected Customer Layer,protected_customer,Customer data after privacy transformations
7,ANA-002,Analytical,Analytics-Safe Customer Layer,analytics_safe_customer,Customer layer without unnecessary direct iden...


# 5. Transformation Registry

In [3]:
transformation_registry = pd.DataFrame([
    ["TR-001","Copy","Direct field passthrough","Source value copied without transformation."],
    ["TR-002","Derived","Is_Blocked","Access_Decision == 'Block' converted to binary indicator."],
    ["TR-003","Governance","Contextual Risk Flags","Risk flags derived from sensitivity, device, failed logins, anomaly, compliance, and governance scores."],
    ["TR-004","Governance","Contextual Risk Score","Boolean risk flags summed into a transparent score."],
    ["TR-005","Governance","Contextual Risk Level","Risk score mapped to LOW, MEDIUM, HIGH, or CRITICAL."],
    ["TR-006","Governance","Baseline Authorization","Role and CRM_Action mapped through proposed RBAC matrix."],
    ["TR-007","Governance","Proposed Access Decision","Priority-based decision engine returns ALLOW, REVIEW, or BLOCK."],
    ["TR-008","Governance","Triggered Rule ID","Decision engine returns the rule responsible for the decision."],
    ["TR-009","Data Quality","Failed Rule Count","Number of failed Data Quality rules calculated per record."],
    ["TR-010","Data Quality","DQ Status","Failed rule count mapped to PASS, WARNING, or FAIL."],
    ["TR-011","Privacy","Customer Token","Customer_ID transformed into deterministic pseudonymous token."],
    ["TR-012","Privacy","Email Masking","Email masked before limited operational use."],
    ["TR-013","Privacy","Phone Masking","Phone reduced to masked value."],
    ["TR-014","Privacy","Age Group","Birth_Date generalized into analytical age band."],
    ["TR-015","Privacy","Data Minimization","Unnecessary direct identifiers removed from analytics-safe layer."]
], columns=["Transformation_ID","Transformation_Type","Transformation_Name","Logic_Description"])
display(transformation_registry)


,Transformation_ID,Transformation_Type,Transformation_Name,Logic_Description
0,TR-001,Copy,Direct field passthrough,Source value copied without transformation.
1,TR-002,Derived,Is_Blocked,Access_Decision == 'Block' converted to binary...
2,TR-003,Governance,Contextual Risk Flags,"Risk flags derived from sensitivity, device, f..."
3,TR-004,Governance,Contextual Risk Score,Boolean risk flags summed into a transparent s...
4,TR-005,Governance,Contextual Risk Level,"Risk score mapped to LOW, MEDIUM, HIGH, or CRI..."
5,TR-006,Governance,Baseline Authorization,Role and CRM_Action mapped through proposed RB...
6,TR-007,Governance,Proposed Access Decision,"Priority-based decision engine returns ALLOW, ..."
7,TR-008,Governance,Triggered Rule ID,Decision engine returns the rule responsible f...
8,TR-009,Data Quality,Failed Rule Count,Number of failed Data Quality rules calculated...
9,TR-010,Data Quality,DQ Status,"Failed rule count mapped to PASS, WARNING, or ..."


# 6. CRM Access Source-to-Target Mapping

In [4]:
access_source_to_target = pd.DataFrame([
    ["Role","raw_access_events.Role","governed_access_events.Role","TR-001"],
    ["Region","raw_access_events.Region","governed_access_events.Region","TR-001"],
    ["Lead_Source","raw_access_events.Lead_Source","governed_access_events.Lead_Source","TR-001"],
    ["CRM_Action","raw_access_events.CRM_Action","governed_access_events.CRM_Action","TR-001"],
    ["Daily_Logins","raw_access_events.Daily_Logins","governed_access_events.Daily_Logins","TR-001"],
    ["Failed_Logins","raw_access_events.Failed_Logins","governed_access_events.Failed_Logins","TR-001"],
    ["Access_Hour","raw_access_events.Access_Hour","governed_access_events.Access_Hour","TR-001"],
    ["Device_Type","raw_access_events.Device_Type","governed_access_events.Device_Type","TR-001"],
    ["Data_Sensitivity","raw_access_events.Data_Sensitivity","governed_access_events.Data_Sensitivity","TR-001"],
    ["Policy_Compliance_Score","raw_access_events.Policy_Compliance_Score","governed_access_events.Policy_Compliance_Score","TR-001"],
    ["Anomaly_Score","raw_access_events.Anomaly_Score","governed_access_events.Anomaly_Score","TR-001"],
    ["Permission_Granted","raw_access_events.Permission_Granted","governed_access_events.Permission_Granted","TR-001"],
    ["Governance_Score","raw_access_events.Governance_Score","governed_access_events.Governance_Score","TR-001"],
    ["Access_Decision","raw_access_events.Access_Decision","governed_access_events.Access_Decision","TR-001"],
    ["Access_Decision","raw_access_events.Access_Decision","governed_access_events.Is_Blocked","TR-002"],
    ["Multiple Source Fields","raw_access_events.*","governed_access_events.Contextual_Risk_Score","TR-004"],
    ["Contextual_Risk_Score","governed_access_events.Contextual_Risk_Score","governed_access_events.Contextual_Risk_Level","TR-005"],
    ["Role + CRM_Action","governed_access_events.Role + CRM_Action","governed_access_events.Baseline_Authorization","TR-006"],
    ["Governance Inputs","governed_access_events.*","governed_access_events.Proposed_Access_Decision","TR-007"],
    ["Governance Inputs","governed_access_events.*","governed_access_events.Triggered_Rule_ID","TR-008"],
    ["DQ Rule Results","dq_record_flags.*","governed_access_events.Failed_Rule_Count","TR-009"],
    ["Failed_Rule_Count","governed_access_events.Failed_Rule_Count","governed_access_events.DQ_Status","TR-010"]
], columns=["Business_Field","Source_Field","Target_Field","Transformation_ID"])
display(access_source_to_target)


,Business_Field,Source_Field,Target_Field,Transformation_ID
0,Role,raw_access_events.Role,governed_access_events.Role,TR-001
1,Region,raw_access_events.Region,governed_access_events.Region,TR-001
2,Lead_Source,raw_access_events.Lead_Source,governed_access_events.Lead_Source,TR-001
3,CRM_Action,raw_access_events.CRM_Action,governed_access_events.CRM_Action,TR-001
4,Daily_Logins,raw_access_events.Daily_Logins,governed_access_events.Daily_Logins,TR-001
5,Failed_Logins,raw_access_events.Failed_Logins,governed_access_events.Failed_Logins,TR-001
6,Access_Hour,raw_access_events.Access_Hour,governed_access_events.Access_Hour,TR-001
7,Device_Type,raw_access_events.Device_Type,governed_access_events.Device_Type,TR-001
8,Data_Sensitivity,raw_access_events.Data_Sensitivity,governed_access_events.Data_Sensitivity,TR-001
9,Policy_Compliance_Score,raw_access_events.Policy_Compliance_Score,governed_access_events.Policy_Compliance_Score,TR-001


# 7. Customer Privacy Source-to-Target Mapping

In [5]:
customer_source_to_target = pd.DataFrame([
    ["Customer_ID","raw_customer.Customer_ID","protected_customer.Customer_Token","TR-011","analytics_safe_customer.Customer_Token"],
    ["First_Name","raw_customer.First_Name","Removed","TR-015","Not exposed"],
    ["Last_Name","raw_customer.Last_Name","Removed","TR-015","Not exposed"],
    ["Email","raw_customer.Email","protected_customer.Email_Masked","TR-012","Not exposed in default analytics layer"],
    ["Phone","raw_customer.Phone","protected_customer.Phone_Masked","TR-013","Not exposed in default analytics layer"],
    ["Birth_Date","raw_customer.Birth_Date","protected_customer.Age_Group","TR-014","analytics_safe_customer.Age_Group"],
    ["State","raw_customer.State","protected_customer.State","TR-001","analytics_safe_customer.State"],
    ["Signup_Date","raw_customer.Signup_Date","protected_customer.Signup_Date","TR-001","analytics_safe_customer.Signup_Date"],
    ["Marketing_Consent","raw_customer.Marketing_Consent","protected_customer.Marketing_Consent","TR-001","analytics_safe_customer.Marketing_Consent"],
    ["Customer_Segment","raw_customer.Customer_Segment","protected_customer.Customer_Segment","TR-001","analytics_safe_customer.Customer_Segment"],
    ["Purchase_Count","raw_customer.Purchase_Count","protected_customer.Purchase_Count","TR-001","analytics_safe_customer.Purchase_Count"],
    ["Total_Revenue","raw_customer.Total_Revenue","protected_customer.Total_Revenue","TR-001","analytics_safe_customer.Total_Revenue"],
    ["Average_Ticket","raw_customer.Average_Ticket","protected_customer.Average_Ticket","TR-001","analytics_safe_customer.Average_Ticket"]
], columns=["Business_Field","Source_Field","Protected_Field","Transformation_ID","Analytics_Target"])
display(customer_source_to_target)


,Business_Field,Source_Field,Protected_Field,Transformation_ID,Analytics_Target
0,Customer_ID,raw_customer.Customer_ID,protected_customer.Customer_Token,TR-011,analytics_safe_customer.Customer_Token
1,First_Name,raw_customer.First_Name,Removed,TR-015,Not exposed
2,Last_Name,raw_customer.Last_Name,Removed,TR-015,Not exposed
3,Email,raw_customer.Email,protected_customer.Email_Masked,TR-012,Not exposed in default analytics layer
4,Phone,raw_customer.Phone,protected_customer.Phone_Masked,TR-013,Not exposed in default analytics layer
5,Birth_Date,raw_customer.Birth_Date,protected_customer.Age_Group,TR-014,analytics_safe_customer.Age_Group
6,State,raw_customer.State,protected_customer.State,TR-001,analytics_safe_customer.State
7,Signup_Date,raw_customer.Signup_Date,protected_customer.Signup_Date,TR-001,analytics_safe_customer.Signup_Date
8,Marketing_Consent,raw_customer.Marketing_Consent,protected_customer.Marketing_Consent,TR-001,analytics_safe_customer.Marketing_Consent
9,Customer_Segment,raw_customer.Customer_Segment,protected_customer.Customer_Segment,TR-001,analytics_safe_customer.Customer_Segment


# 8. Data Quality Rule Lineage

In [6]:
dq_lineage = pd.DataFrame([
    ["DQ-COMP-001","User_ID","Completeness"],["DQ-COMP-002","Role","Completeness"],
    ["DQ-COMP-003","CRM_Action","Completeness"],["DQ-COMP-004","Permission_Granted","Completeness"],
    ["DQ-COMP-005","Access_Decision","Completeness"],["DQ-VAL-001","Role","Validity"],
    ["DQ-VAL-002","CRM_Action","Validity"],["DQ-VAL-003","Device_Type","Validity"],
    ["DQ-VAL-004","Access_Hour","Validity"],["DQ-VAL-005","Data_Sensitivity","Validity"],
    ["DQ-VAL-006","Anomaly_Score","Validity"],["DQ-VAL-007","Policy_Compliance_Score","Validity"],
    ["DQ-VAL-008","Governance_Score","Validity"],["DQ-VAL-009","Daily_Logins","Validity"],
    ["DQ-VAL-010","Failed_Logins","Validity"],["DQ-CON-001","Role","Consistency"],
    ["DQ-CON-001","CRM_Action","Consistency"],["DQ-CON-002","Permission_Granted","Consistency"],
    ["DQ-CON-002","Access_Decision","Consistency"]
], columns=["DQ_Rule_ID","Field_Name","DQ_Dimension"])
display(dq_lineage)


,DQ_Rule_ID,Field_Name,DQ_Dimension
0,DQ-COMP-001,User_ID,Completeness
1,DQ-COMP-002,Role,Completeness
2,DQ-COMP-003,CRM_Action,Completeness
3,DQ-COMP-004,Permission_Granted,Completeness
4,DQ-COMP-005,Access_Decision,Completeness
5,DQ-VAL-001,Role,Validity
6,DQ-VAL-002,CRM_Action,Validity
7,DQ-VAL-003,Device_Type,Validity
8,DQ-VAL-004,Access_Hour,Validity
9,DQ-VAL-005,Data_Sensitivity,Validity


# 9. Governance Rule Lineage

In [7]:
governance_rule_lineage = pd.DataFrame([
    ["AUTH-001","Permission_Granted","Authorization"],["AUTH-002","Role","Authorization"],
    ["AUTH-002","CRM_Action","Authorization"],["AUTH-003","Role","Authorization"],
    ["AUTH-003","CRM_Action","Authorization"],["CTX-001","Data_Sensitivity","Contextual Risk"],
    ["CTX-001","Device_Type","Contextual Risk"],["CTX-001","Failed_Logins","Contextual Risk"],
    ["CTX-001","Anomaly_Score","Contextual Risk"],["CTX-001","Policy_Compliance_Score","Contextual Risk"],
    ["CTX-001","Governance_Score","Contextual Risk"],["CTX-002","Data_Sensitivity","Contextual Risk"],
    ["CTX-002","Device_Type","Contextual Risk"],["CTX-002","Anomaly_Score","Contextual Risk"],
    ["CTX-004","Data_Sensitivity","Contextual Risk"],["CTX-004","Failed_Logins","Contextual Risk"]
], columns=["Governance_Rule_ID","Field_Name","Rule_Category"])
display(governance_rule_lineage)


,Governance_Rule_ID,Field_Name,Rule_Category
0,AUTH-001,Permission_Granted,Authorization
1,AUTH-002,Role,Authorization
2,AUTH-002,CRM_Action,Authorization
3,AUTH-003,Role,Authorization
4,AUTH-003,CRM_Action,Authorization
5,CTX-001,Data_Sensitivity,Contextual Risk
6,CTX-001,Device_Type,Contextual Risk
7,CTX-001,Failed_Logins,Contextual Risk
8,CTX-001,Anomaly_Score,Contextual Risk
9,CTX-001,Policy_Compliance_Score,Contextual Risk


# 10. Privacy Control Lineage

In [8]:
privacy_control_lineage = pd.DataFrame([
    ["PRIV-001","First_Name","Remove from analytics"],
    ["PRIV-001","Last_Name","Remove from analytics"],
    ["PRIV-002","Customer_ID","Pseudonymize"],
    ["PRIV-003","Email","Mask"],
    ["PRIV-004","Phone","Mask"],
    ["PRIV-005","Birth_Date","Generalize"],
    ["PRIV-006","Marketing_Consent","Keep / Restrict"],
    ["PRIV-007","Multiple Fields","Minimize"]
], columns=["Privacy_Control_ID","Field_Name","Treatment"])
display(privacy_control_lineage)


,Privacy_Control_ID,Field_Name,Treatment
0,PRIV-001,First_Name,Remove from analytics
1,PRIV-001,Last_Name,Remove from analytics
2,PRIV-002,Customer_ID,Pseudonymize
3,PRIV-003,Email,Mask
4,PRIV-004,Phone,Mask
5,PRIV-005,Birth_Date,Generalize
6,PRIV-006,Marketing_Consent,Keep / Restrict
7,PRIV-007,Multiple Fields,Minimize


# 11. Analytical Output Registry

In [9]:
analytical_output_registry = pd.DataFrame([
    ["KPI-001","Blocked Access Rate","Access Governance","Is_Blocked / Access_Decision"],
    ["KPI-002","Proposed Block Rate","Access Governance","Proposed_Access_Decision"],
    ["KPI-003","Access Under Review","Access Governance","Proposed_Access_Decision"],
    ["KPI-004","Critical Risk Events","Risk Governance","Contextual_Risk_Level"],
    ["KPI-005","High Sensitivity Access","Data Governance","Data_Sensitivity"],
    ["KPI-006","BYOD Access Events","Security","Device_Type"],
    ["KPI-007","Data Quality Score","Data Quality","DQ Rule Results"],
    ["KPI-008","Failed DQ Records","Data Quality","DQ_Status / Failed_Rule_Count"],
    ["KPI-009","Rule Trigger Frequency","Governance Operations","Triggered_Rule_ID"],
    ["KPI-010","Marketing Consent Rate","Privacy","Marketing_Consent"],
    ["KPI-011","Restricted Field Count","Privacy","Privacy Classification"],
    ["KPI-012","Analytics-Safe Customer Count","Privacy","Customer_Token"]
], columns=["Output_ID","Output_Name","Domain","Primary_Input"])
display(analytical_output_registry)


,Output_ID,Output_Name,Domain,Primary_Input
0,KPI-001,Blocked Access Rate,Access Governance,Is_Blocked / Access_Decision
1,KPI-002,Proposed Block Rate,Access Governance,Proposed_Access_Decision
2,KPI-003,Access Under Review,Access Governance,Proposed_Access_Decision
3,KPI-004,Critical Risk Events,Risk Governance,Contextual_Risk_Level
4,KPI-005,High Sensitivity Access,Data Governance,Data_Sensitivity
5,KPI-006,BYOD Access Events,Security,Device_Type
6,KPI-007,Data Quality Score,Data Quality,DQ Rule Results
7,KPI-008,Failed DQ Records,Data Quality,DQ_Status / Failed_Rule_Count
8,KPI-009,Rule Trigger Frequency,Governance Operations,Triggered_Rule_ID
9,KPI-010,Marketing Consent Rate,Privacy,Marketing_Consent


# 12. End-to-End Access Traceability Matrix

In [10]:
access_traceability = pd.DataFrame([
    ["Permission_Granted","AUTH-001","TR-007","Proposed_Access_Decision","KPI-002 / KPI-003"],
    ["Role","AUTH-002 / AUTH-003","TR-006 + TR-007","Baseline_Authorization / Proposed_Access_Decision","KPI-002 / KPI-003 / KPI-009"],
    ["CRM_Action","AUTH-002 / AUTH-003","TR-006 + TR-007","Baseline_Authorization / Proposed_Access_Decision","KPI-002 / KPI-003 / KPI-009"],
    ["Data_Sensitivity","CTX-001 / CTX-002 / CTX-004","TR-003 + TR-004 + TR-005 + TR-007","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004 / KPI-005"],
    ["Device_Type","CTX-001 / CTX-002","TR-003 + TR-004 + TR-005 + TR-007","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004 / KPI-006"],
    ["Failed_Logins","CTX-001 / CTX-004","TR-003 + TR-004 + TR-005 + TR-007","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004"],
    ["Anomaly_Score","CTX-001 / CTX-002","TR-003 + TR-004 + TR-005 + TR-007","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004"],
    ["Policy_Compliance_Score","CTX-001","TR-003 + TR-004 + TR-005 + TR-007","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004"],
    ["Governance_Score","CTX-001","TR-003 + TR-004 + TR-005 + TR-007","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004"],
    ["Access_Decision","DQ-CON-002","TR-002","Is_Blocked","KPI-001"]
], columns=["Source_Field","Governance_or_DQ_Rule","Transformation","Derived_or_Target_Field","Analytical_Output"])
display(access_traceability)


,Source_Field,Governance_or_DQ_Rule,Transformation,Derived_or_Target_Field,Analytical_Output
0,Permission_Granted,AUTH-001,TR-007,Proposed_Access_Decision,KPI-002 / KPI-003
1,Role,AUTH-002 / AUTH-003,TR-006 + TR-007,Baseline_Authorization / Proposed_Access_Decision,KPI-002 / KPI-003 / KPI-009
2,CRM_Action,AUTH-002 / AUTH-003,TR-006 + TR-007,Baseline_Authorization / Proposed_Access_Decision,KPI-002 / KPI-003 / KPI-009
3,Data_Sensitivity,CTX-001 / CTX-002 / CTX-004,TR-003 + TR-004 + TR-005 + TR-007,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004 / KPI-005
4,Device_Type,CTX-001 / CTX-002,TR-003 + TR-004 + TR-005 + TR-007,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004 / KPI-006
5,Failed_Logins,CTX-001 / CTX-004,TR-003 + TR-004 + TR-005 + TR-007,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004
6,Anomaly_Score,CTX-001 / CTX-002,TR-003 + TR-004 + TR-005 + TR-007,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004
7,Policy_Compliance_Score,CTX-001,TR-003 + TR-004 + TR-005 + TR-007,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004
8,Governance_Score,CTX-001,TR-003 + TR-004 + TR-005 + TR-007,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004
9,Access_Decision,DQ-CON-002,TR-002,Is_Blocked,KPI-001


# 13. End-to-End Privacy Traceability Matrix

In [11]:
privacy_traceability = pd.DataFrame([
    ["Customer_ID","PRIV-002","TR-011","Customer_Token","KPI-012"],
    ["First_Name","PRIV-001 / PRIV-007","TR-015","Removed","Not exposed"],
    ["Last_Name","PRIV-001 / PRIV-007","TR-015","Removed","Not exposed"],
    ["Email","PRIV-003 / PRIV-007","TR-012 + TR-015","Email_Masked / Removed","Not exposed by default"],
    ["Phone","PRIV-004 / PRIV-007","TR-013 + TR-015","Phone_Masked / Removed","Not exposed by default"],
    ["Birth_Date","PRIV-005","TR-014","Age_Group","Customer segmentation"],
    ["Marketing_Consent","PRIV-006","TR-001","Marketing_Consent","KPI-010"],
    ["State","PRIV-007","TR-001","State","Customer analytics"],
    ["Customer_Segment","PRIV-007","TR-001","Customer_Segment","Customer analytics"],
    ["Purchase_Count","PRIV-007","TR-001","Purchase_Count","Customer analytics"],
    ["Total_Revenue","PRIV-007","TR-001","Total_Revenue","Customer analytics"]
], columns=["Raw_Field","Privacy_Control","Transformation","Analytics_Field","Analytical_Use"])
display(privacy_traceability)


,Raw_Field,Privacy_Control,Transformation,Analytics_Field,Analytical_Use
0,Customer_ID,PRIV-002,TR-011,Customer_Token,KPI-012
1,First_Name,PRIV-001 / PRIV-007,TR-015,Removed,Not exposed
2,Last_Name,PRIV-001 / PRIV-007,TR-015,Removed,Not exposed
3,Email,PRIV-003 / PRIV-007,TR-012 + TR-015,Email_Masked / Removed,Not exposed by default
4,Phone,PRIV-004 / PRIV-007,TR-013 + TR-015,Phone_Masked / Removed,Not exposed by default
5,Birth_Date,PRIV-005,TR-014,Age_Group,Customer segmentation
6,Marketing_Consent,PRIV-006,TR-001,Marketing_Consent,KPI-010
7,State,PRIV-007,TR-001,State,Customer analytics
8,Customer_Segment,PRIV-007,TR-001,Customer_Segment,Customer analytics
9,Purchase_Count,PRIV-007,TR-001,Purchase_Count,Customer analytics


# 14. Lineage Coverage Metrics

In [12]:
access_fields_expected = {
    "Permission_Granted","Role","CRM_Action","Data_Sensitivity","Device_Type",
    "Failed_Logins","Anomaly_Score","Policy_Compliance_Score","Governance_Score","Access_Decision"
}
access_fields_documented = set(access_traceability["Source_Field"])

privacy_fields_expected = {
    "Customer_ID","First_Name","Last_Name","Email","Phone","Birth_Date",
    "Marketing_Consent","State","Customer_Segment","Purchase_Count","Total_Revenue"
}
privacy_fields_documented = set(privacy_traceability["Raw_Field"])

lineage_kpis = pd.DataFrame({
    "Metric":[
        "Access Governance Lineage Coverage %",
        "Privacy Lineage Coverage %",
        "Registered Transformations",
        "Registered Analytical Outputs"
    ],
    "Value":[
        len(access_fields_expected & access_fields_documented)/len(access_fields_expected)*100,
        len(privacy_fields_expected & privacy_fields_documented)/len(privacy_fields_expected)*100,
        len(transformation_registry),
        len(analytical_output_registry)
    ]
})
display(lineage_kpis.round(2))


,Metric,Value
0,Access Governance Lineage Coverage %,100.0
1,Privacy Lineage Coverage %,100.0
2,Registered Transformations,15.0
3,Registered Analytical Outputs,12.0


# 15. Rule-to-KPI Impact Analysis

In [13]:
rule_to_kpi = pd.DataFrame([
    ["AUTH-001","Permission_Granted","Proposed_Access_Decision","KPI-002 / KPI-003"],
    ["AUTH-002","Role + CRM_Action","Baseline_Authorization / Proposed_Access_Decision","KPI-002 / KPI-003 / KPI-009"],
    ["AUTH-003","Role + CRM_Action","Baseline_Authorization / Proposed_Access_Decision","KPI-002 / KPI-003 / KPI-009"],
    ["CTX-001","Multiple Risk Fields","Contextual_Risk_Level / Proposed_Access_Decision","KPI-004 / KPI-009"],
    ["CTX-002","Data_Sensitivity + Device_Type + Anomaly_Score","Proposed_Access_Decision","KPI-002 / KPI-004 / KPI-009"],
    ["CTX-004","Data_Sensitivity + Failed_Logins","Proposed_Access_Decision","KPI-003 / KPI-009"],
    ["DQ-CON-002","Permission_Granted + Access_Decision","DQ_Status","KPI-007 / KPI-008"],
    ["PRIV-002","Customer_ID","Customer_Token","KPI-012"],
    ["PRIV-006","Marketing_Consent","Marketing_Consent","KPI-010"]
], columns=["Rule_or_Control_ID","Input_Fields","Affected_Target","Potentially_Affected_Output"])
display(rule_to_kpi)


,Rule_or_Control_ID,Input_Fields,Affected_Target,Potentially_Affected_Output
0,AUTH-001,Permission_Granted,Proposed_Access_Decision,KPI-002 / KPI-003
1,AUTH-002,Role + CRM_Action,Baseline_Authorization / Proposed_Access_Decision,KPI-002 / KPI-003 / KPI-009
2,AUTH-003,Role + CRM_Action,Baseline_Authorization / Proposed_Access_Decision,KPI-002 / KPI-003 / KPI-009
3,CTX-001,Multiple Risk Fields,Contextual_Risk_Level / Proposed_Access_Decision,KPI-004 / KPI-009
4,CTX-002,Data_Sensitivity + Device_Type + Anomaly_Score,Proposed_Access_Decision,KPI-002 / KPI-004 / KPI-009
5,CTX-004,Data_Sensitivity + Failed_Logins,Proposed_Access_Decision,KPI-003 / KPI-009
6,DQ-CON-002,Permission_Granted + Access_Decision,DQ_Status,KPI-007 / KPI-008
7,PRIV-002,Customer_ID,Customer_Token,KPI-012
8,PRIV-006,Marketing_Consent,Marketing_Consent,KPI-010


# 16. Impact Analysis Example

If the threshold for `High_Anomaly_Flag` changes:

```text
Anomaly_Score
   ↓
High_Anomaly_Flag
   ↓
Contextual_Risk_Score
   ↓
Contextual_Risk_Level
   ↓
Proposed_Access_Decision
   ↓
Critical Risk Events / Proposed Block Rate / Access Under Review / Rule Trigger Frequency
```

This is why lineage supports change management, not only documentation.


# 17. Proposed Lineage Governance Operating Model

| Activity | Proposed Responsibility |
|---|---|
| Define critical lineage requirements | Data Governance |
| Maintain source-to-target mappings | Data Engineering |
| Validate business lineage | Data Steward |
| Validate security-rule dependencies | Information Security |
| Validate privacy-control lineage | Privacy / DPO |
| Maintain BI metric lineage | BI / Analytics |
| Approve changes to critical mappings | Data Owner |


# 18. Lineage Monitoring KPIs

Future monitoring can include:
- Lineage Coverage %
- Critical Data Elements with Lineage %
- Governance Rules with Documented Inputs %
- Privacy Controls with Documented Inputs %
- Analytical Outputs with Traceable Sources %
- Registered Transformations
- Unmapped Fields
- Unmapped KPIs
- Broken Lineage Links
- Last Lineage Review Date


# 19. Exportable Lineage Artifacts

Reusable artifacts:
- `layer_inventory`
- `transformation_registry`
- `access_source_to_target`
- `customer_source_to_target`
- `dq_lineage`
- `governance_rule_lineage`
- `privacy_control_lineage`
- `analytical_output_registry`
- `access_traceability`
- `privacy_traceability`
- `rule_to_kpi`


# 20. Findings to Document

## Architecture

- **Number of documented layers:** 8 documented data layers covering source, raw, governed, privacy-protected, and analytical structures for both access-governance and customer-data flows.

- **Number of registered transformations:** 15 transformations.

- **Number of analytical outputs:** 12 registered analytical outputs / KPIs.

## Access Governance Lineage

- **Coverage:** 100% lineage coverage for the access-governance fields included in the traceability framework.

- **Fields without lineage:** 0 fields in scope are without documented lineage.

- **Rules with most downstream dependencies:** `AUTH-002`, `AUTH-003`, and `CTX-002` have the broadest direct downstream KPI impact in the registered impact analysis, each potentially affecting three analytical outputs.

## Privacy Lineage

- **Coverage:** 100% lineage coverage for the privacy-related fields included in the framework.

- **Fields removed from analytics:** `First_Name` and `Last_Name` are explicitly removed. `Email` and `Phone` are also not exposed in the default analytics-safe layer after their protected representations are created.

- **Fields transformed before analytics:** `Customer_ID` is pseudonymized into `Customer_Token`; `Birth_Date` is generalized into `Age_Group`; `Email` and `Phone` are masked before being excluded from the default analytics layer.

## Impact Analysis

- **Most connected source fields:** `Data_Sensitivity`, `Role`, and `CRM_Action` are among the most connected access-governance source fields. `Data_Sensitivity` supports three contextual governance rules, while `Role` and `CRM_Action` each support multiple authorization rules and downstream governance KPIs.

- **Most connected governance rules:** `AUTH-002`, `AUTH-003`, and `CTX-002` have the largest direct registered downstream KPI reach. `CTX-001` is also highly relevant because it aggregates multiple upstream risk fields and affects both contextual-risk and rule-monitoring outputs.

- **KPIs most sensitive to upstream changes:** `KPI-009 — Rule Trigger Frequency` has the broadest dependency across governance rules, followed by `KPI-002 — Proposed Block Rate` and `KPI-003 — Access Under Review`.

## Governance Conclusions

1. **The lineage framework provides end-to-end traceability from source fields through transformations and governance or privacy controls to analytical outputs, with 100% documented coverage for the access-governance and privacy flows in scope.**

2. **Impact analysis shows that changes to highly connected fields or governance rules can propagate across several derived attributes and KPIs, making lineage essential for controlled change management and regression assessment.**

3. **Lineage should be maintained as an operational governance asset rather than static documentation, with ownership, transformation changes, control dependencies, and downstream KPI impacts reviewed whenever source structures or governance rules change.**


# 21. Limitations

1. Lineage is manually documented.
2. No automated metadata scanner or lineage platform is connected.
3. SQL statements and orchestration jobs do not exist in this prototype.
4. Power BI outputs are planned rather than physically connected.
5. Transformation logic is conceptual rather than captured from production pipelines.
6. Review dates and lineage version history are not implemented yet.
7. The dataset is synthetic.


# 22. Next Step — Stewardship & Governance Operating Model

## Stage 8 — Stewardship & Governance Operating Model

Planned outputs:
- governance roles and responsibilities;
- Data Owner vs. Data Steward responsibilities;
- RACI matrix;
- issue-management workflow;
- Data Quality escalation process;
- policy exception workflow;
- metadata review process;
- governance committee structure;
- rule review cadence;
- governance decision log.
